## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### `ORIG cross DEST`
#### AMPL Formulation

In [8]:
%%ampl_eval
reset;              # clear previous model/data

set ORIG;                           # origin cities
set DEST;                           # destination cities
set LINKS = ORIG cross DEST;        # all (origin, destination) pairs

param cost1 {LINKS} >= 0;               # Form 1: named set using cross operator
param cost2 {ORIG, DEST} >= 0;          # Form 2: anonymous comma-syntax in an indexing expression
param cost3 {i in ORIG, j in DEST} >= 0;# Form 3: named dummy indices in comma-syntax

# All three declare a parameter indexed over all (origin, destination) pairs.
# Forms 1 and 2 are preferred when the index set is referenced more than once.

In [9]:
%%ampl_eval
data;

# Origin and destination sets
set ORIG := O1 O2 O3;
set DEST := D1 D2 D3;

# Transportation cost per (origin, destination)
param cost1:
       D1  D2  D3 :=
O1     2   4   5
O2     3   1   7
O3     6   2   3;

param cost2:
       D1  D2  D3 :=
O1     3   5   6
O2     4   1   7
O3     7   2   3;

param cost3:
       D1  D2  D3 :=
O1     4   6   7
O2     5   1   7
O3     9   2   3;

display cost1, cost2, cost3;                     # show cost1, cost2, cost3         

:     cost1 cost2 cost3    :=
O1 D1    2     3     4
O1 D2    4     5     6
O1 D3    5     6     7
O2 D1    3     4     5
O2 D2    1     1     1
O2 D3    7     7     7
O3 D1    6     7     9
O3 D2    2     2     2
O3 D3    3     3     3
;



#### amplpy Formulation

In [11]:
import pandas as pd

# Sets
ampl.set['ORIG'] = ['O1', 'O2', 'O3']
ampl.set['DEST'] = ['D1', 'D2', 'D3']

cost1 = pd.DataFrame({      # Cost matrix 1
    'D1': [2, 3, 6],
    'D2': [4, 1, 2],
    'D3': [5, 7, 3]
}, index=['O1', 'O2', 'O3'])

ampl.param['cost1'].set_values(cost1)

cost2 = pd.DataFrame({      # Cost matrix 2
    'D1': [3, 4, 7],
    'D2': [5, 1, 2],
    'D3': [6, 7, 3]
}, index=['O1', 'O2', 'O3'])

ampl.param['cost2'].set_values(cost2)

cost3 = pd.DataFrame({      # Cost matrix 3
    'D1': [4, 5, 9],
    'D2': [6, 1, 2],
    'D3': [7, 7, 3]
}, index=['O1', 'O2', 'O3'])

ampl.param['cost3'].set_values(cost3)

# Display results (equivalent to AMPL display)
print("cost1:\n", ampl.param['cost1'].get_values())
print("cost2:\n", ampl.param['cost2'].get_values())
print("cost3:\n", ampl.param['cost3'].get_values())

cost1:
    index0       index1    |    cost1    
    'O1'           'D1'     |      2      
    'O1'           'D2'     |      4      
    'O1'           'D3'     |      5      
    'O2'           'D1'     |      3      
    'O2'           'D2'     |      1      
    'O2'           'D3'     |      7      
    'O3'           'D1'     |      6      
    'O3'           'D2'     |      2      
    'O3'           'D3'     |      3      

cost2:
    index0       index1    |    cost2    
    'O1'           'D1'     |      3      
    'O1'           'D2'     |      5      
    'O1'           'D3'     |      6      
    'O2'           'D1'     |      4      
    'O2'           'D2'     |      1      
    'O2'           'D3'     |      7      
    'O3'           'D1'     |      7      
    'O3'           'D2'     |      2      
    'O3'           'D3'     |      3      

cost3:
    index0       index1    |    cost3    
    'O1'           'D1'     |      4      
    'O1'           'D2'     |     